# 5.1 Code Brief: Institutional Pattern Discovery with Unsupervised Learning

## Key Concepts

- Unsupervised — no target variable, discover structure instead of predicting an outcome.
- The consistent pipeline: **Scale -> Find k (Elbow + Silhouette) -> Fit K-Means -> PCA visualization -> Profile clusters -> Interpret**.
- All data in this notebook is synthetically generated inline (`np.random...`) — fully self-contained, no external CSV needed.
- Four case studies, same pipeline each time: student entry segmentation, program portfolio, student pathways, course bottleneck detection.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visualization defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully!")
print("This module uses matplotlib/seaborn for static, publication-ready visuals.")

## Case Study 1: Student Entry Segmentation

In [ ]:
# --- Generate synthetic student entry data ---
n = 1000

entry_df = pd.DataFrame({
    'HS_GPA':             np.random.normal(3.2, 0.45, n).clip(2.0, 4.0),
    'HS_MATH_GPA':        np.random.normal(3.0, 0.55, n).clip(1.5, 4.0),
    'HS_ENGL_GPA':        np.random.normal(3.1, 0.50, n).clip(1.5, 4.0),
    'UNITS_ATTEMPTED_1':  np.random.choice([9, 12, 13, 14, 15, 16, 17, 18], n,
                                            p=[0.05, 0.15, 0.10, 0.15, 0.25, 0.15, 0.10, 0.05]),
    'FIRST_GEN':          np.random.binomial(1, 0.42, n),
    'PELL_ELIGIBLE':      np.random.binomial(1, 0.38, n),
    'DISTANCE_FROM_CAMPUS': np.random.exponential(25, n).clip(1, 200).round(1)
})

print(f"Dataset: {entry_df.shape[0]:,} students × {entry_df.shape[1]} variables")
entry_df.describe().round(2)

In [ ]:
scaler = StandardScaler()
entry_scaled = scaler.fit_transform(entry_df)

print("Before scaling:")
print(f"  HS_GPA range:    {entry_df['HS_GPA'].min():.1f} – {entry_df['HS_GPA'].max():.1f}")
print(f"  DISTANCE range:  {entry_df['DISTANCE_FROM_CAMPUS'].min():.1f} – {entry_df['DISTANCE_FROM_CAMPUS'].max():.1f}")
print(f"\nAfter scaling (mean ≈ 0, std ≈ 1):")
print(f"  HS_GPA range:    {entry_scaled[:, 0].min():.2f} – {entry_scaled[:, 0].max():.2f}")
print(f"  DISTANCE range:  {entry_scaled[:, 6].min():.2f} – {entry_scaled[:, 6].max():.2f}")

In [ ]:
# Elbow method + Silhouette scores
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(entry_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(entry_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
axes[0].legend()

# Silhouette plot
axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nSilhouette scores: " + ", ".join([f"k={k}: {s:.3f}" for k, s in zip(K_range, silhouettes)]))

In [ ]:
km_entry = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
entry_df['Cluster'] = km_entry.fit_predict(entry_scaled)

print(f"Silhouette Score: {silhouette_score(entry_scaled, entry_df['Cluster']):.3f}")
print(f"\nCluster Sizes:")
print(entry_df['Cluster'].value_counts().sort_index())

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
entry_pca = pca.fit_transform(entry_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(entry_pca[:, 0], entry_pca[:, 1],
                     c=entry_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Student Entry Segments (PCA Projection)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance")
print(f"PC2 explains {pca.explained_variance_ratio_[1]:.1%} of variance")
print(f"Together: {sum(pca.explained_variance_ratio_[:2]):.1%}")

In [ ]:
profile = entry_df.groupby('Cluster').mean().round(2)
profile['Count'] = entry_df['Cluster'].value_counts().sort_index()

print("=" * 70)
print("CLUSTER PROFILES — Student Entry Segmentation")
print("=" * 70)
print(profile.to_string())
print("=" * 70)

# Heatmap of cluster profiles (z-scored for comparison)
profile_z = (profile.drop(columns='Count') - profile.drop(columns='Count').mean()) / profile.drop(columns='Count').std()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(profile_z, annot=profile.drop(columns='Count').values, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=1, ax=ax)
ax.set_title('Cluster Profiles (color = z-score, numbers = raw means)')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

## Case Study 2: Program Portfolio Segmentation

In [ ]:
# --- Generate synthetic program portfolio data ---
n_prog = 300

prog_df = pd.DataFrame({
    'ENROLLMENT':        np.random.lognormal(5.5, 0.7, n_prog).clip(50, 2000).astype(int),
    'RETENTION_RATE':    np.random.beta(8, 3, n_prog).clip(0.40, 0.98).round(3),
    'GRADUATION_RATE':   np.random.beta(5, 4, n_prog).clip(0.20, 0.90).round(3),
    'FACULTY_RATIO':     np.random.normal(22, 7, n_prog).clip(8, 40).round(1),
    'COST_PER_STUDENT':  np.random.lognormal(9.5, 0.5, n_prog).clip(5000, 50000).round(0),
    'JOB_PLACEMENT_RATE': np.random.beta(7, 3, n_prog).clip(0.50, 0.98).round(3)
})

print(f"Dataset: {prog_df.shape[0]} programs × {prog_df.shape[1]} variables")
prog_df.describe().round(2)

In [ ]:
# Scale → Elbow + Silhouette → Fit K-Means → PCA → Profile
scaler_prog = StandardScaler()
prog_scaled = scaler_prog.fit_transform(prog_df)

# Optimal k
inertias_p, sils_p = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(prog_scaled)
    inertias_p.append(km.inertia_)
    sils_p.append(silhouette_score(prog_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(2, 11), inertias_p, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method — Programs')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7)

axes[1].plot(range(2, 11), sils_p, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette — Programs')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7)
plt.tight_layout(); plt.show()

In [ ]:
# Fit with k=4
km_prog = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
prog_df['Cluster'] = km_prog.fit_predict(prog_scaled)

# PCA visualization
pca_prog = PCA(n_components=2, random_state=RANDOM_STATE)
prog_pca = pca_prog.fit_transform(prog_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(prog_pca[:, 0], prog_pca[:, 1],
                     c=prog_df['Cluster'], cmap='Set2', alpha=0.6, s=40, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_prog.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_prog.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Program Portfolio Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

# Profile
profile_p = prog_df.groupby('Cluster').mean().round(2)
profile_p['Count'] = prog_df['Cluster'].value_counts().sort_index()
print("=" * 80)
print("CLUSTER PROFILES — Program Portfolio Segmentation")
print("=" * 80)
print(profile_p.to_string())

## Case Study 3: Student Pathway Segmentation

In [ ]:
# --- Generate synthetic pathway data ---
n_path = 1000

pathway_df = pd.DataFrame({
    'GPA_1':                 np.random.normal(2.8, 0.7, n_path).clip(0.0, 4.0).round(2),
    'GPA_2':                 np.random.normal(2.9, 0.65, n_path).clip(0.0, 4.0).round(2),
    'DFW_RATE_1':            np.random.beta(2, 8, n_path).clip(0.0, 1.0).round(3),
    'DFW_RATE_2':            np.random.beta(2, 8, n_path).clip(0.0, 1.0).round(3),
    'UNITS_COMPLETED_RATIO': np.random.beta(8, 2, n_path).clip(0.3, 1.0).round(3)
})

print(f"Dataset: {pathway_df.shape[0]:,} students × {pathway_df.shape[1]} variables")
pathway_df.describe().round(2)

In [ ]:
# Full pipeline: Scale → Elbow/Silhouette → K-Means → PCA → Profile
scaler_path = StandardScaler()
path_scaled = scaler_path.fit_transform(pathway_df)

# Optimal k
sils_path = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_path.append(silhouette_score(path_scaled, km.fit_predict(path_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_path, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis — Student Pathways')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
ax.legend(); plt.tight_layout(); plt.show()

# Fit k=4
km_path = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
pathway_df['Cluster'] = km_path.fit_predict(path_scaled)

# PCA + scatter
pca_path = PCA(n_components=2, random_state=RANDOM_STATE)
path_pca = pca_path.fit_transform(path_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(path_pca[:, 0], path_pca[:, 1],
                     c=pathway_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_path.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_path.explained_variance_ratio_[1]:.1%})')
ax.set_title('Student Pathway Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

# Profile
profile_path = pathway_df.groupby('Cluster').mean().round(3)
profile_path['Count'] = pathway_df['Cluster'].value_counts().sort_index()
print("=" * 70)
print("CLUSTER PROFILES — Student Pathways")
print("=" * 70)
print(profile_path.to_string())

## Final Capstone: Course Bottleneck Detection

In [ ]:
# --- Generate synthetic course section data ---
n_courses = 1000

course_df = pd.DataFrame({
    'ENROLLMENT':        np.random.lognormal(3.5, 0.5, n_courses).clip(15, 500).astype(int),
    'DFW_RATE':          np.random.beta(2, 7, n_courses).clip(0.01, 0.70).round(3),
    'PASS_RATE':         np.random.beta(7, 2, n_courses).clip(0.30, 0.99).round(3),
    'AVG_GPA':           np.random.normal(2.7, 0.5, n_courses).clip(0.5, 4.0).round(2),
    'REPEAT_RATE':       np.random.beta(1.5, 8, n_courses).clip(0.0, 0.50).round(3),
    'AVG_REPEAT_DELAY':  np.random.exponential(1.5, n_courses).clip(0.5, 6.0).round(1),
    'SECTION_SIZE':      np.random.choice([25, 30, 35, 40, 50, 60, 80, 100, 150, 200], n_courses),
    'PCT_FIRST_YEAR':    np.random.beta(3, 5, n_courses).round(3),
    'PCT_STEM':          np.random.beta(3, 4, n_courses).round(3),
    'INSTRUCTOR_RATING': np.random.normal(3.8, 0.6, n_courses).clip(1.0, 5.0).round(1),
    'PREREQUISITE_COUNT': np.random.choice([0, 1, 2, 3, 4, 5], n_courses, p=[0.15, 0.30, 0.25, 0.15, 0.10, 0.05]),
    'IS_GATEWAY':        np.random.binomial(1, 0.30, n_courses),
    'UNITS':             np.random.choice([1, 2, 3, 4, 5], n_courses, p=[0.05, 0.10, 0.50, 0.30, 0.05])
})

print(f"Dataset: {course_df.shape[0]:,} course sections × {course_df.shape[1]} variables")
course_df.describe().round(2)

In [ ]:
# Full pipeline: Scale → Elbow/Silhouette → K-Means → PCA → Profile
scaler_c = StandardScaler()
course_scaled = scaler_c.fit_transform(course_df)

# Optimal k
sils_c = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_c.append(silhouette_score(course_scaled, km.fit_predict(course_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_c, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis — Course Sections')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
ax.legend(); plt.tight_layout(); plt.show()

# Fit k=4
km_course = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
course_df['Cluster'] = km_course.fit_predict(course_scaled)

# PCA visualization
pca_c = PCA(n_components=2, random_state=RANDOM_STATE)
course_pca = pca_c.fit_transform(course_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(course_pca[:, 0], course_pca[:, 1],
                     c=course_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_c.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_c.explained_variance_ratio_[1]:.1%})')
ax.set_title('Course Section Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

In [ ]:
# Cluster Profile
profile_c = course_df.groupby('Cluster').mean().round(3)
profile_c['Count'] = course_df['Cluster'].value_counts().sort_index()
print("=" * 100)
print("CLUSTER PROFILES — Course Bottleneck Detection")
print("=" * 100)
print(profile_c.to_string())
print("=" * 100)

# Heatmap
profile_cz = (profile_c.drop(columns='Count') - profile_c.drop(columns='Count').mean()) / profile_c.drop(columns='Count').std()
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_cz, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0, linewidths=1, ax=ax)
ax.set_title('Course Cluster Profiles (z-scored — red = concerning, green = favorable)')
ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

## Risk Prioritization Index

Composite risk score: 0.4 × normalized DFW rate + 0.3 × normalized repeat rate + 0.3 × normalized inverse pass rate.

In [ ]:
# Risk Prioritization Index
course_df['RISK_INDEX'] = (
    0.4 * (course_df['DFW_RATE'] / course_df['DFW_RATE'].max()) +
    0.3 * (course_df['REPEAT_RATE'] / course_df['REPEAT_RATE'].max()) +
    0.3 * ((1 - course_df['PASS_RATE']) / (1 - course_df['PASS_RATE']).max())
).round(3)

# Top 20 highest-risk sections
top_risk = course_df.nlargest(20, 'RISK_INDEX')[
    ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE', 'IS_GATEWAY', 'RISK_INDEX', 'Cluster']
]

print("=" * 80)
print("TOP 20 HIGHEST-RISK COURSE SECTIONS")
print("=" * 80)
print(top_risk.to_string())

# Risk distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
course_df.boxplot(column='RISK_INDEX', by='Cluster', ax=axes[0])
axes[0].set_title('Risk Index by Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Risk Index')
plt.sca(axes[0]); plt.title('Risk Index by Cluster')

# Histogram
for c in sorted(course_df['Cluster'].unique()):
    axes[1].hist(course_df[course_df['Cluster'] == c]['RISK_INDEX'],
                 alpha=0.5, bins=20, label=f'Cluster {c}')
axes[1].set_xlabel('Risk Index'); axes[1].set_ylabel('Count')
axes[1].set_title('Risk Index Distribution by Cluster')
axes[1].legend()
plt.tight_layout(); plt.show()

## The Consistent Pattern

```
Scale -> Find k -> Fit K-Means -> PCA Visualization -> Profile -> Interpret
```

## Key Takeaways
- Always scale before K-Means (distance-based algorithm)
- Elbow + Silhouette to pick k
- `df.groupby('Cluster').mean()` is the institutional-insight step
- Risk Index = weighted composite score for prioritization